# MLflow experiments and model comparison

Run `python -m src.train` from the project root first. This notebook reads a completed batch instead of creating duplicate runs.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Image

root = Path.cwd()
if not (root / 'src').exists():
    root = root.parent
sys.path.insert(0, str(root))
output = root / 'reports' / 'forecasts'
latest = json.loads((output / 'latest.json').read_text())
batch = output / latest['batch_id']
summary = json.loads((batch / 'summary.json').read_text())

## Compare validation runs

In [ ]:
comparison = pd.read_csv(batch / 'comparison.csv')
display(comparison[['run_name', 'model_type', 'rmse', 'mae', 'wape', 'r2', 'nonconverged_stores']])
display(Image(filename=str(batch / 'comparison.png')))

In [ ]:
from src.tracking import configure_tracking, EXPERIMENT, REGISTERED_MODEL
client = configure_tracking(root)
experiment = client.get_experiment_by_name(EXPERIMENT)
runs = client.search_runs([experiment.experiment_id], filter_string=f"tags.batch_id = '{latest['batch_id']}'", order_by=['metrics.rmse ASC'])
[(run.info.run_name, run.data.metrics.get('rmse')) for run in runs]

## Selected model and held-out test

The lowest validation RMSE among converged candidates determines the winner; the test is not used for ranking.

In [ ]:
display(summary['best_run'])
display(summary['test_metrics'])
display(summary['splits'])
display(Image(filename=str(batch / 'selected_test' / 'test_forecast.png')))

## Registry

In [ ]:
version = client.get_model_version(REGISTERED_MODEL, summary['version'])
print('Model:', version.name, '| Version:', version.version, '| Stage:', version.current_stage)
print('Run ID:', version.run_id)

## Interpretation

After running the project, explain which family improved over the naive baseline, how the validation and test errors differ, and whether holiday peaks were difficult to predict. Review the registered version before promotion.